## 第13章 Web内容爬取

### 1.webbrowser模块

- 打开网页：`webbrowser.open(url)`。

### 2.requests模块

- 下载网页内容：`requests.get(url)`。
    - 获取响应状态码：`response.status_code`。
    - 获取响应内容：`response.text`。
- 检查访问错误：`response.raise_for_status()`。

In [ ]:
import requests

# 下载网页内容
resp = requests.get("https://automatetheboringstuff.com/files/rj.txt")

# 检查访问是否出错
try:
    resp.raise_for_status()
except Exception as e:
    print(f'There was a problem: {e}')
    exit()

# 保存下载的文件
with open("./res/RomeoAndJuliet.txt", "wb") as f:
    for chunk in resp.iter_content(100000):
        f.write(chunk)

### 3.使用API

In [ ]:
import requests

key = "SGshgn3Amyu9UVVKW"  # 网上提供的API密钥，随时可能失效
city= "杭州"

# 获取城市ID
resp = requests.get(f"https://api.seniverse.com/v3/location/search.json?key={key}&q={city}")
data = resp.json()
id = data["results"][0]["id"]

# 获取当前天气信息
resp = requests.get(f"https://api.seniverse.com/v3/weather/now.json?key={key}&location={id}&language=zh-Hans&unit=c")
data = resp.json()
print(f"{city}当前的天气：{data['results'][0]['now']['text']}， 温度：{data['results'][0]['now']['temperature']}℃")

### 4.使用BeautifulSoup模块

In [ ]:
import requests, bs4

res = requests.get("https://autbor.com/example3.html")
res.raise_for_status()

soup = bs4.BeautifulSoup(res.text, "html.parser")
eles = soup.select("#author")
print(f"Element: {eles[0]}")
print(f"Text: {eles[0].text}")
print(f"Attributes: {eles[0].attrs}")
print(f"ID: {eles[0].attrs['id']}")

In [ ]:
# 下载XKCD漫画
import requests, bs4, os, time

# 初始化变量
MAX_DOWNLOADS= 10  # 最大下载次数
num_downloads= 0
url= "https://xkcd.com"
os.makedirs("./res/xkcd", exist_ok=True)  # 创建存储目录

while not url.endswith("#") and num_downloads < MAX_DOWNLOADS:
    print(f"Downloading page {url}...")
    res = requests.get(url)
    res.raise_for_status()

    # 获取漫画图片URL
    soup = bs4.BeautifulSoup(res.text, "html.parser")
    comic_ele = soup.select("#comic img")
    if len(comic_ele) > 0:
        # 下载漫画图片
        comic_url = f"https:{comic_ele[0].get('src')}"
        print(f'Downloading image {comic_url}...')
        res = requests.get(comic_url)
        res.raise_for_status()
    else:
        print("Could not find comic image!")
        continue

    # 保存漫画图片
    image_file = open(os.path.join("./res/xkcd", os.path.basename(comic_url)), "wb")
    for chunk in res.iter_content(100000):
        image_file.write(chunk)
    image_file.close()

    # 获取上一页链接
    prev_link = soup.select("a[rel='prev']")
    if len(prev_link) > 0:
        url = f"https://xkcd.com{prev_link[0].get('href')}"
        num_downloads += 1
        time.sleep(1)
    else:
        break

print("Done!")

### 5.使用Selenium模块

- 导入Selenium模块： `from selenium import webdriver`。
- 创建浏览器对象： `browser = webdriver.Chrome()`。
- 访问目标网站： `browser.get("https://www.baidu.com")`。
- 点击浏览器按钮：`browser.back()`、`browser.forward()`、`browser.refresh()`、`browser.quit()`。
- 查找页面元素：`browser.find_element(By.ID, "kw")`、`browser.find_elements(By.CLASS_NAME, "su")`。
- 点击页面元素：`browser.find_element(By.ID, "su").click()`。
- 输入文本：`browser.find_element(By.ID, "kw").send_keys("Python")`。

### 6.使用Playwright模块

- 导入Playwright模块：`from playwright.sync_api import sync_playwright`。
- 创建Playwright对象：`playwright = sync_playwright().start()`。
- 创建浏览器对象：`browser = playwright.chromium.launch(headless=False)`。
- 创建页面对象：`page = browser.new_page()`。
- 访问目标网站：`page.goto("https://www.baidu.com")`。
- 点击浏览器按钮：`page.go_back()`、`page.go_forward()`、`page.reload()`、`page.close()`。
- 查找页面元素：`page.locator("#kw")`。
- 获取某个元素：`page.locator("#kw").nth()`。
- 点击页面元素：`page.click("#su")` 或 `page.locator("#su").click()`。
- 输入文本：`page.locator("#kw").fill("Python")`。
- 关闭浏览器对象：`browser.close()`。
- 退出Playwright：`playwright.stop()`。